### Import Dependencies

In [4]:
import openai
import pandas as pd
import cohere

from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import VectorParams, Distance, SparseVectorParams, Modifier, PayloadSchemaType, PointStruct, Document, Prefetch, FusionQuery

In [3]:
from dotenv import load_dotenv

load_dotenv("../../.env")

True

### Retrieval

In [8]:
query = "Can I get a tablet?"

In [5]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [6]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

In [7]:
def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )
        ],
        query=models.RrfQuery(rrf=models.Rrf(weights=[3,1])),
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [11]:
results = retrieve_data(query, k=20)

In [12]:
results

{'retrieved_context_ids': ['B0B157WDDJ',
  'B0BN58Z4YX',
  'B0BSD3QK7M',
  'B0B159KDFP',
  'B09RHF4L45',
  'B0BGHBL86V',
  'B0BHVH4D37',
  'B0C7DCS2KW',
  'B09WR36NK8',
  'B0BCFYCXRH',
  'B0BT83RRJ2',
  'B0B58CPFWY',
  'B0BWRZ1HRD',
  'B0B6ZZH83Y',
  'B07Y36DDYM',
  'B09RN3KN5C',
  'B0BNKF9J6J',
  'B0B96TTN6T',
  'B08SC3C9MC',
  'B09PYV6B4B'],
 'retrieved_context': ["G-TiDE Kids Tablet, 7 inch Tablet for Kids, 32GB+2GB Kids Learning Tablet, 5MP Dual Camera HD, Parental Control App- KLAP, Toddler Tablet Case, WiFi Tablets Shoulder Straps, Pink 🤗【Explore More Fun on Klap】G-TiDE Klap kids tablet is designed for learning and playing. This tablet for kids offers various creative contents such as brain training, painting, gaming, kids TV, etc. Learning while playing is better for kids to know the world. With a 32GB bigger storage, which can be extended to 128GB (micro SD card not included), this kids tablet is perfect for children aged 3-7 years old. You could get more educational apps from 

### Reranking

In [14]:
cohere_client = cohere.ClientV2()

In [17]:
to_rerank = results["retrieved_context"]

In [18]:
to_rerank

["G-TiDE Kids Tablet, 7 inch Tablet for Kids, 32GB+2GB Kids Learning Tablet, 5MP Dual Camera HD, Parental Control App- KLAP, Toddler Tablet Case, WiFi Tablets Shoulder Straps, Pink 🤗【Explore More Fun on Klap】G-TiDE Klap kids tablet is designed for learning and playing. This tablet for kids offers various creative contents such as brain training, painting, gaming, kids TV, etc. Learning while playing is better for kids to know the world. With a 32GB bigger storage, which can be extended to 128GB (micro SD card not included), this kids tablet is perfect for children aged 3-7 years old. You could get more educational apps from Google Play Store (GMS). 🤗【Kids-proof Case & Eye-protection Screen】G-TiDE's exclusive kids tablet case is made of impact-resistant EVA material. When your kid is playing outdoors with the tablet or accidentally dropping the tablet on the ground at home, it protects the kids tablet from bumps and minor drops. This kids tablet comes with an adjustable shoulder strap, 

In [19]:
query

'Can I get a tablet?'

In [20]:
response = cohere_client.rerank(
    model="rerank-v4.0-pro",
    query=query,
    documents=to_rerank,
    top_n=20
)

In [21]:
response

V2RerankResponse(id='b2e13ba5-6ebe-4718-b620-9508c33cd92d', results=[V2RerankResponseResultsItem(index=4, relevance_score=0.8699799), V2RerankResponseResultsItem(index=15, relevance_score=0.8632072), V2RerankResponseResultsItem(index=6, relevance_score=0.8502698), V2RerankResponseResultsItem(index=2, relevance_score=0.84826964), V2RerankResponseResultsItem(index=19, relevance_score=0.8462476), V2RerankResponseResultsItem(index=8, relevance_score=0.84317327), V2RerankResponseResultsItem(index=9, relevance_score=0.8374073), V2RerankResponseResultsItem(index=3, relevance_score=0.83473045), V2RerankResponseResultsItem(index=0, relevance_score=0.8331075), V2RerankResponseResultsItem(index=7, relevance_score=0.8320184), V2RerankResponseResultsItem(index=17, relevance_score=0.82927084), V2RerankResponseResultsItem(index=12, relevance_score=0.8208144), V2RerankResponseResultsItem(index=5, relevance_score=0.80415994), V2RerankResponseResultsItem(index=1, relevance_score=0.77294225), V2RerankRes

In [22]:
reranked_results = [to_rerank[result.index] for result in response.results]

In [23]:
reranked_results

['plimpton Android Tablet 10 inch - 8 Core Processor, 4GB RAM 64GB Storage, 1.8M Length USB-C Cable, Android 11, 1920 x 1200 FHD IPS, 13MP Sony Rear Camera, 6000mAh Battery, 2.4G+5G WiFi Tablets PC High Performance Tablet 10.1 inch: Features a faster 1.8GHz Octa-Core CPU and 4GB RAM, you can not only obtain seamless multitasking and hyper-fast content streaming experience, but also can view social media, watch videos, play games, complete homework or finish work. With a 64GB internal storage and support a SD card expanded to 512GB(NOT included), no longer worry about insufficient memory.★★Please note this is a Wi-Fi tablet, not compatible with SIM card 2022 Latest Android 11 Tablets: Equipped with the latest Android 11 system (GMS certified). Automatic subtitleing, smart reply, notification "bubble", dark theme, eye protection mode etc. It also launches apps 20% faster than Android 10 OS. New control spaces and more privacy features allow you to create a rich user central expression. Y